# 🎯 11 · Finding particles: template matching, difference-of-Gaussians and their pitfalls
### Locating molecules in a noisy, crowded tomogram — with and without a template — and asking whether a "hit" is real

**BiologyToResearch** · Part C — Classical computational cryo-ET (2009–2019) · Module 11 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/11_particle_picking_classical.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2–3 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-image, scikit-learn |
| **Prerequisites** | Modules 03 (Fourier transform, cross-correlation), 04 (precision/recall, hypothesis tests), 09 (in silico cell). Helpful: 06 (CTF), 07 (missing wedge) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Pei *et al.* 2016 (crowded-cell simulation + DoG picking); Wang *et al.* 2018 (pseudo-structures for template-matching statistics); Zhou *et al.* 2018 (feature-decomposition saliency); Li *et al.* 2019 (Faster-RCNN mitochondria); Uddin *et al.* 2025 (book chapter, feature detection) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods. They illustrate the core idea of each
> paper; they do **not** reproduce the papers' numbers (our tomograms, noise model, molecules and SNR scale all
> differ). **Nothing is stored in the repository:** molecules are real Protein Data Bank (PDB) models, downloaded the
> first time a notebook needs them and cached on your computer (offline, code-built stand-ins of the same size and
> symmetry are used instead); all images and data are simulated by code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Implement** template matching as a fast, locally normalized cross-correlation (FFT) maximized over a set of rotations,
   and extract particle positions with **non-maximum suppression**.
2. **Evaluate** a picker honestly: one-to-one matching, the Pei *et al.* true-positive rule $|x-c|/R\le 0.5$, precision,
   recall, F1 and precision–recall curves.
3. **Implement** difference-of-Gaussians (DoG) picking exactly as in Pei *et al.* (2016) — $\sigma_2=1.1\,\sigma_1$, the
   threshold rule $T=m+t\,(M-m)/20$ — and **explain** why the best $\sigma$ depends on particle size ($r\approx\sqrt3\,\sigma$).
4. **Measure** how crowding and noise degrade picking, and reproduce the paper's qualitative trend (recall falls with crowding).
5. **Explain** why a "best-matching template" is not evidence by itself, and **compute** a Monte Carlo p-value from a null
   distribution built with pseudo-structures (the idea of Wang *et al.* 2018).
6. **Describe** two other classical detectors from the lab — supervoxel saliency with robust PCA (Zhou *et al.* 2018) and
   Faster-RCNN organelle detection (Li *et al.* 2019) — and when each is appropriate.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, molecular zoo | 5 min |
| 1 · A tomogram with an answer key | build an in silico cell; define how to score a picker | 15 min |
| 2 · Template matching | FFT correlation × rotations, NMS, PR curves, rotation sampling, look-alike molecules | 30 min |
| 3 · Difference of Gaussians | the Pei *et al.* recipe; scale vs size; crowding and noise | 30 min |
| 4 · Is a match real? | pseudo-structures → null distribution → p-values | 20 min |
| 5 · Saliency without templates | supervoxels + robust PCA | 10 min |
| 6 · Beyond blobs | organelle detection, a picker comparison table | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A cellular tomogram of a bacterium is a 3D image of a few hundred nanometres of cytoplasm holding thousands of
ribosomes, chaperones, proteasomes and metabolic enzymes. Before anyone can average ribosomes to see their translation
states, or count proteasomes near an aggregate, someone must answer a deceptively simple question: **where are the
molecules?** A 200 × 500 × 500 nm tomogram at 1 nm voxels has 50 million voxels; each molecule occupies a few thousand of
them, and at a signal-to-noise ratio of ~0.01–0.1 they are barely visible. Finding the candidates — **particle picking** —
is the entry door of every subtomogram study, and its errors (missed molecules, picked noise, picked look-alikes) propagate
into every later result.

---
## 0 · Environment setup

Run the cells in order (Shift + Enter). The first cell reports the hardware; everything here runs on a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("skimage", "scikit-image"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.fft as sfft
from scipy import ndimage, stats
from scipy.spatial import cKDTree

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| SciPy FFT backend | seed", SEED)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on its
own. You met them in Modules 02 (rotations), 03 (Fourier masks), 06 (CTF), 07 (missing wedge) and 09 (crowded cell).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def support_radius(t: np.ndarray, level: float = 0.1) -> float:
    """Radius (voxels) of the smallest centred sphere containing all voxels above ``level``·max."""
    pts = np.argwhere(t > level * t.max())
    if len(pts) == 0:
        return t.shape[0] / 2
    c = (np.array(t.shape) - 1) / 2
    return float(np.linalg.norm(pts - c, axis=1).max()) + 1.0


def make_crowded_cell(templates: dict, counts: dict, shape=(48, 160, 160), rng=None,
                      membrane: bool = True, tries_per_particle: int = 400):
    """Pack randomly rotated templates into a volume without overlap (bounding spheres).

    A simple rejection-sampling stand-in for the packing used by Pei et al. (2016)
    and Liu et al. (2020).  Particles are placed in random class order so that every
    class gets a fair chance.  Returns (clean_volume, particles) where each particle
    is dict(cls, center_zyx, R).  Optionally adds a spherical membrane vesicle.
    """
    rng = rng or np.random.default_rng(0)
    vol = np.zeros(shape, np.float32)
    occ_c, occ_r, particles = [], [], []
    if membrane:  # a vesicle: thin shell, ~4 nm thick lipid bilayer at 1 nm/voxel
        zc, yc, xc = shape[0] / 2, shape[1] * 0.78, shape[2] * 0.22
        rad = min(shape[1], shape[2]) * 0.16
        g = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
        d = np.sqrt((g[0] - zc)**2 + (g[1] - yc)**2 + (g[2] - xc)**2)
        vol += 0.6 * np.exp(-((d - rad)**2) / (2 * 1.5**2))
        occ_c.append(np.array([zc, yc, xc])); occ_r.append(rad + 3)
    radii = {cls: support_radius(templates[cls], level=0.35) for cls in counts}
    order = [cls for cls, n in counts.items() for _ in range(n)]
    rng.shuffle(order)
    for cls in order:
        t, rad_t = templates[cls], radii[cls]
        for _ in range(tries_per_particle):
            c = np.array([rng.uniform(rad_t, s - rad_t) if s > 2 * rad_t else s / 2 for s in shape])
            if any(np.linalg.norm(c - oc) < rad_t + orr for oc, orr in zip(occ_c, occ_r)):
                continue
            R = random_rotation(rng)
            sub = rotate_volume(t, R)
            h = np.array(sub.shape) // 2
            lo = np.round(c).astype(int) - h
            sl_v, sl_s = [], []
            for a in range(3):
                a0, a1 = max(lo[a], 0), min(lo[a] + sub.shape[a], shape[a])
                sl_v.append(slice(a0, a1)); sl_s.append(slice(a0 - lo[a], a1 - lo[a]))
            vol[tuple(sl_v)] += sub[tuple(sl_s)]
            occ_c.append(c); occ_r.append(rad_t)
            particles.append(dict(cls=cls, center_zyx=c, R=R))
            break
    return vol, particles


def simulate_tomogram(clean: np.ndarray, rng, snr: float = 0.05, tilt_max_deg: float = 60.0,
                      ctf_pixel_A: float | None = 10.0, defocus_um: float = 5.0) -> np.ndarray:
    """Fast tomogram simulation: (optional 3D CTF-like filter) → missing wedge → noise.

    The CTF is applied isotropically here (a simplification of per-tilt CTF).
    """
    v = clean
    if ctf_pixel_A:
        ctf = ctf_2d(clean.shape, ctf_pixel_A, defocus_um=defocus_um, kV=300, B=0)
        v = apply_fourier_mask(v, -ctf)          # sign flip: dense objects appear bright
    v = apply_fourier_mask(v, missing_wedge_mask(v.shape, tilt_max_deg))
    return add_noise_snr(v, snr, rng)
print("🧰 toolkit loaded")

### The molecular zoo

We use the course's four "star" molecules (Module 00): the 70S **ribosome** (4V4R, 2.3 MDa), the 20S **proteasome**
(3J9I, a D7 barrel), the **singly capped proteasome** (6UTJ — the same barrel plus a cap) and **RuBisCO** (1BXN, the 290 kDa L₄S₄ half).
We keep two box sizes: 32³ (for building the cell) and 40³ (for Section 4, where molecules must fit with some margin).

In [ ]:
# ── Data: build and prepare the zoo ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
NAMES = {pid: ZOO[pid]["name"] for pid in CAST}
t0 = time.time()
raw_maps = {}
for pid in CAST:
    vol, voxel_A, source = zoo_map(pid)
    raw_maps[pid] = vol
    print(f"  {pid.upper()}  {NAMES[pid]:26s} raw {str(vol.shape):14s} @ {voxel_A:.0f} Å/voxel  ← {source}")
zoo32 = {pid: prepare_map(raw_maps[pid], size=32, sigma=1.0) for pid in CAST}
zoo40 = {pid: prepare_map(raw_maps[pid], size=40, sigma=1.0) for pid in CAST}
# bounding-sphere radius at 0.2 × max density, as in Pei et al. (2016)
R_BS = {pid: support_radius(zoo32[pid], level=0.2) for pid in CAST}
for pid in CAST:
    print(f"  {pid.upper()} bounding-sphere radius ≈ {R_BS[pid]:.1f} nm")
print(f"done in {time.time() - t0:.1f} s")

---
## 1 · A tomogram with an answer key

> 📍 **Where we are.** Module 09 built *in silico cells* — crowded volumes of molecules with known positions — and
> Modules 06–07 explained how the microscope blurs them (CTF) and removes a wedge of Fourier space. Module 03 showed that
> cross-correlation, computed with FFTs, finds where a pattern sits in an image. This module turns those pieces into
> **particle pickers** and, just as importantly, into an honest way of **scoring** them.

Real tomograms have no answer key, so pickers are developed and benchmarked on simulated ones — exactly the motivation of
Pei *et al.* (2016). We pack ribosomes, proteasomes, capped proteasomes and RuBisCO into a 160 × 160 × 64 nm box with a
membrane vesicle, then image it (CTF + ±60° missing wedge + noise at **SNR 0.03**).

In [ ]:
# ── 1.1 Build and image the in silico cell ──
COUNTS = {"4v4r": 12, "3j9i": 10, "6utj": 8, "1bxn": 24}
SNR_MAIN = 0.03
t0 = time.time()
clean, particles = make_crowded_cell(zoo32, COUNTS, shape=(64, 160, 160), rng=np.random.default_rng(SEED))
tomo = simulate_tomogram(clean, np.random.default_rng(SEED + 1), snr=SNR_MAIN, tilt_max_deg=60)
gt_cls = np.array([p["cls"] for p in particles])
gt_xyz = np.array([p["center_zyx"] for p in particles])          # NB: stored as (z, y, x)
gt_rad = np.array([R_BS[c] for c in gt_cls])
occupancy = np.sum(4 / 3 * np.pi * gt_rad**3) / clean.size
print(f"placed {len(particles)} particles in {time.time() - t0:.1f} s; "
      f"bounding-sphere occupancy ≈ {100 * occupancy:.0f} % of the volume")
for pid in CAST:
    print(f"  {NAMES[pid]:26s}: {np.sum(gt_cls == pid)}")

In [ ]:
# ── 1.2 Look at the tomogram with the answer key ──
MARK = {"4v4r": ("o", "tab:red"), "3j9i": ("s", "tab:blue"), "6utj": ("D", "tab:cyan"), "1bxn": ("^", "tab:green")}


def draw_truth(ax, z, slab=6, scale=1.0, offset=0.0, s=110):
    """Overlay ground-truth particles whose centres lie within ±slab voxels of slice z (coordinates /scale)."""
    for c, (zz, yy, xx) in zip(gt_cls, gt_xyz):
        if abs(zz - z) <= slab:
            m, col = MARK[c]
            ax.scatter((xx - offset) / scale, (yy - offset) / scale, marker=m, facecolors="none",
                       edgecolors=col, s=s, lw=1.3)


z_rib = gt_xyz[gt_cls == "4v4r", 0]
z_mid = int(max(range(16, 48), key=lambda z: np.sum(np.abs(z_rib - z) <= 5)))   # a slice through several ribosomes
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.6))
axes[0].imshow(clean[z_mid]); axes[0].set_title(f"true density, z = {z_mid} nm")
axes[1].imshow(ndimage.gaussian_filter(tomo[z_mid], 1)); axes[1].set_title(f"simulated tomogram (SNR {SNR_MAIN}), lightly smoothed")
for ax in axes:
    draw_truth(ax, z_mid)
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
for pid, (m, col) in MARK.items():
    axes[0].scatter([], [], marker=m, facecolors="none", edgecolors=col, label=NAMES[pid])
axes[0].legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The left panel is the answer key; the right panel is what a picker sees. Big ribosomes (red
circles) are still faintly visible, RuBisCO (green triangles) almost disappears, and the membrane vesicle (bottom left) is a
strong, *non-particle* signal that will fool naive pickers.

### 1.3 How do we score a picker?

A picker outputs a list of positions. We compare them with the true centres using the **true-positive rule of Pei *et al.*
(2016)**: a picked position $x_p$ is a hit for particle $g$ (centre $C_g$, bounding-sphere radius $R_g$) if

$$S=\frac{\lVert x_p-C_g\rVert}{R_g}\le 0.5 ,$$

*in words:* the pick must land within half a particle radius of the true centre. Each particle can be claimed by **one**
pick only (extra picks on the same particle count as false positives), so we match greedily from the smallest $S$ upward.
Then (Module 04)

$$\text{precision}=\frac{TP}{TP+FP},\qquad \text{recall}=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}.$$

*In words:* precision = "how many of my picks are real?", recall = "how many real particles did I find?", and $F_1$ is
their harmonic mean (high only if both are high).

In [ ]:
# ── 1.3 One-to-one matching with the Pei et al. rule ──
def match_pei(picks_zyx, true_zyx, true_radius, s_max=0.5):
    """Greedy one-to-one matching: a pick hits a particle if |x_p − C_g| / R_g ≤ s_max.

    Returns (metrics dict with precision/recall/f1/tp/n_picks, boolean 'found' per true particle,
    boolean 'hit' per pick).
    """
    picks = np.asarray(picks_zyx, float).reshape(-1, 3)
    found, hit = np.zeros(len(true_zyx), bool), np.zeros(len(picks), bool)
    if len(picks) == 0 or len(true_zyx) == 0:
        return dict(precision=0.0, recall=0.0, f1=0.0, tp=0, n_picks=len(picks)), found, hit
    S = np.linalg.norm(picks[:, None, :] - np.asarray(true_zyx)[None], axis=-1) / np.asarray(true_radius)[None]
    for i, j in zip(*np.unravel_index(np.argsort(S, axis=None), S.shape)):
        if S[i, j] > s_max:
            break
        if not hit[i] and not found[j]:
            hit[i] = found[j] = True
    tp = int(found.sum())
    p, r = tp / len(picks), tp / len(true_zyx)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-12), tp=tp, n_picks=len(picks)), found, hit


# sanity check: perfect picks, then picks shifted by 0.6 R (should fail the rule)
m_ok, _, _ = match_pei(gt_xyz, gt_xyz, gt_rad)
m_bad, _, _ = match_pei(gt_xyz + np.c_[0.6 * gt_rad, 0 * gt_rad, 0 * gt_rad], gt_xyz, gt_rad)
print("perfect picks:", {k: round(v, 2) for k, v in m_ok.items()})
print("picks 0.6 R off:", {k: round(v, 2) for k, v in m_bad.items()})
assert m_ok["f1"] == 1.0 and m_bad["tp"] == 0, "matching rule is broken"

---
## 2 · Template matching: "where does this shape fit best?"

🧫 If you already know what a ribosome looks like (from a PDB model or a previous average), you can slide a copy of it —
a **template** — through the tomogram and ask, at every position, *how well does the local density resemble the
template?* Because every ribosome in the cell points in a different direction, you must also try many **rotations**
of the template. This is the method of the first "visual proteomics" studies (Frangakis *et al.* 2002; Beck *et al.* 2009).

### 2.1 The score: locally normalized cross-correlation

For a template $t$ restricted to a spherical mask $M$ with $n$ voxels, first normalize it inside the mask to zero mean and
unit variance: $t'(y)$. The score at position $x$ is the Pearson correlation between $t'$ and the tomogram patch around $x$:

$$c(x)=\frac{1}{n\,\sigma_I(x)}\sum_{y\in M} t'(y)\,I(x+y),\qquad
\sigma_I^2(x)=\frac1n\sum_{y\in M}I(x+y)^2-\Big(\frac1n\sum_{y\in M}I(x+y)\Big)^2 .$$

*In words:* multiply the normalized template with the local patch, add up, and divide by the patch's own standard deviation
so that bright-but-unrelated regions (membranes, ice) do not win just by being bright. Both sums over $y$ are
**correlations**, so they are computed for *all* positions $x$ at once with FFTs (Module 03): that is the "fast local
correlation function" of Roseman (2003). One more detail matters in cryo-ET: the template must be **imaged the same way
as the tomogram** — we apply the same CTF and missing-wedge filter to each rotated template before correlating.

💡 **Binning.** Template matching is usually run on a **binned** tomogram (2 × 2 × 2 voxels averaged into one): 8× fewer
voxels, 8× faster FFTs, and a ribosome still spans ~14 voxels. We bin 1 nm → 2 nm.

In [ ]:
# ── 2.1 Template-matching helpers ──
def bin2(vol):
    """Average 2×2×2 blocks: halves every dimension (e.g. 1 nm → 2 nm voxels)."""
    s = [n // 2 * 2 for n in vol.shape]
    v = vol[:s[0], :s[1], :s[2]]
    return v.reshape(s[0] // 2, 2, s[1] // 2, 2, s[2] // 2, 2).mean(axis=(1, 3, 5)).astype(np.float32)


def sphere_mask(n, radius):
    """Binary sphere of the given radius (voxels) centred in an n³ box."""
    g = np.arange(n) - (n - 1) / 2
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    return (np.sqrt(x * x + y * y + z * z) <= radius).astype(np.float32)


def imaging_filter(n, pixel_A, tilt_max_deg=60.0, defocus_um=5.0):
    """Fourier filter of our simulated microscope on an n³ grid: sign-flipped CTF × missing wedge."""
    return (-ctf_2d((n,) * 3, pixel_A, defocus_um=defocus_um)) * missing_wedge_mask((n,) * 3, tilt_max_deg)


def kernel_at_origin(small, shape):
    """Embed a small centred kernel in a zero array of ``shape`` with its centre moved to index (0, 0, 0)."""
    out = np.zeros(shape, np.float32)
    m = small.shape[0]
    out[:m, :m, :m] = small
    return np.roll(out, (-(m // 2),) * 3, axis=(0, 1, 2))


def template_match(tomo, template, rotations, mask, filt, snapshots=()):
    """Fast local normalized cross-correlation, maximized over rotations.

    Returns (best score map, index of best rotation per voxel, {k: best map after the first k rotations}).
    """
    shp, n = tomo.shape, mask.sum()
    FI = sfft.rfftn(tomo, workers=1)
    FM = np.conj(sfft.rfftn(kernel_at_origin(mask, shp), workers=1))
    mu = sfft.irfftn(FI * FM, s=shp, workers=1) / n                       # local mean under the mask
    mu2 = sfft.irfftn(sfft.rfftn(tomo**2, workers=1) * FM, s=shp, workers=1) / n
    sd = np.sqrt(np.maximum(mu2 - mu**2, 1e-12))                          # local standard deviation
    best = np.full(shp, -np.inf, np.float32)
    arg = np.zeros(shp, np.int32)
    snaps = {}
    for i, R in enumerate(rotations):
        t = apply_fourier_mask(rotate_volume(template, R), filt)          # rotate, then "image" the template
        inside = t[mask > 0]
        t = (t - inside.mean()) / (inside.std() + 1e-12) * mask            # zero mean, unit variance in the mask
        FT = np.conj(sfft.rfftn(kernel_at_origin(t, shp), workers=1))
        score = sfft.irfftn(FI * FT, s=shp, workers=1) / (n * sd)
        better = score > best
        best[better], arg[better] = score[better], i
        if i + 1 in snapshots:
            snaps[i + 1] = best.copy()
    return best, arg, snaps


def nms_peaks(score, min_dist, threshold=-np.inf):
    """Non-maximum suppression: local maxima above threshold, then greedily drop peaks closer than min_dist
    to a stronger peak. Returns (peaks [k, 3] in voxel indices, values), strongest first."""
    is_max = score == ndimage.maximum_filter(score, size=3, mode="nearest")
    pk = np.argwhere(is_max & (score > threshold))
    vals = score[tuple(pk.T)]
    order = np.argsort(-vals)
    pk, vals = pk[order], vals[order]
    if len(pk) == 0:
        return pk, vals
    keep = np.ones(len(pk), bool)
    tree = cKDTree(pk)
    for i in range(len(pk)):
        if keep[i]:
            for j in tree.query_ball_point(pk[i], min_dist):
                if j > i:
                    keep[j] = False
    return pk[keep], vals[keep]


print("helpers ready: bin2, sphere_mask, imaging_filter, template_match, nms_peaks")

In [ ]:
# ── 2.2 Search the binned tomogram for ribosomes with 300 random rotations ──
tomo_b = bin2(tomo)                                   # 32 × 80 × 80 voxels at 2 nm
tpl_b = {pid: bin2(zoo32[pid]) for pid in CAST}       # 16³ templates at 2 nm
FILT16 = imaging_filter(16, pixel_A=20.0)
N_ROT = 300
ROTS = [random_rotation(np.random.default_rng(1000 + i)) for i in range(N_ROT)]
SNAPS = (1, 3, 10, 30, 100, 300)


def to_full_res(peaks_b):
    """Voxel index in the 2× binned map → coordinate in the 1 nm tomogram (centre of the 2×2×2 block)."""
    return np.asarray(peaks_b, float) * 2 + 0.5


mask_rib = sphere_mask(16, support_radius(tpl_b["4v4r"], 0.2) + 0.5)
t0 = time.time()
score_rib, arg_rib, snaps_rib = template_match(tomo_b, tpl_b["4v4r"], ROTS, mask_rib, FILT16, snapshots=SNAPS)
t_tm = time.time() - t0
print(f"template matching: {N_ROT} rotations on a {tomo_b.shape} map in {t_tm:.1f} s "
      f"({1000 * t_tm / N_ROT:.0f} ms per rotation)")
peaks_rib, vals_rib = nms_peaks(score_rib, min_dist=R_BS["4v4r"] / 2)
print(f"{len(peaks_rib)} candidate peaks after NMS; top-15 scores:", np.round(vals_rib[:15], 2))

🔍 **What just happened?** For each of 300 rotations we rotated the ribosome template, imaged it, and correlated it with
the whole tomogram in one FFT; each voxel kept its best score over rotations (and which rotation gave it). **Non-maximum
suppression (NMS)** then turned the score map into a short list of peaks: keep only local maxima, and drop any peak closer
than a minimum distance (here half a ribosome radius) to a stronger one. Let us look at the score map.

In [ ]:
# ── 2.3 The score map and the picks ──
zb = z_mid // 2                                            # binned slice index (same height as Figure 1.2)
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.4))
axes[0].imshow(tomo_b[zb]); axes[0].set_title(f"binned tomogram (2 nm voxels), z ≈ {2 * zb} nm")
im = axes[1].imshow(score_rib[zb], cmap="viridis", vmin=0); axes[1].set_title("best ribosome score over 300 rotations")
plt.colorbar(im, ax=axes[1], fraction=0.046, label="local correlation")
top = peaks_rib[:int(np.sum(gt_cls == "4v4r"))]
for ax in axes:
    draw_truth(ax, 2 * zb + 0.5, slab=6, scale=2, offset=0.5, s=80)
    sel = np.abs(top[:, 0] - zb) <= 3
    ax.scatter(top[sel, 2], top[sel, 1], marker="+", color="yellow", s=60, lw=1.5)
    ax.set_xlabel("x (2-nm voxels)"); ax.set_ylabel("y (2-nm voxels)")
plt.tight_layout(); plt.show()

🔍 The yellow crosses are the top-ranked peaks (as many as there are ribosomes); red circles are true ribosomes. The score
is high on ribosomes, intermediate on the other large molecules, and low elsewhere — even though the vesicle is bright,
the *local normalization* stops it from dominating.

### 2.4 Precision–recall curves

A picker's output is a ranked list. Every cut-off (keep the top $k$ peaks, or keep scores above a threshold) gives one
(precision, recall) pair; sweeping the cut-off traces the **precision–recall (PR) curve**. A perfect picker reaches
the top-right corner; the best $F_1$ along the curve is a convenient single number.

In [ ]:
# ── 2.4 PR curve for ribosome template matching ──
def pr_curve(peaks_zyx, true_zyx, true_radius, k_max=None):
    """Precision/recall when keeping the top-k ranked picks, for k = 1..k_max."""
    k_max = min(k_max or len(peaks_zyx), len(peaks_zyx))
    rows = [match_pei(peaks_zyx[:k], true_zyx, true_radius)[0] for k in range(1, k_max + 1)]
    return np.array([r["precision"] for r in rows]), np.array([r["recall"] for r in rows]), np.array([r["f1"] for r in rows])


is_rib = gt_cls == "4v4r"
SNR_HARD = 0.005                                     # a much noisier image of the SAME cell
tomo_hard_b = bin2(simulate_tomogram(clean, np.random.default_rng(SEED + 1), snr=SNR_HARD))
t0 = time.time()
score_hard, _, snaps_hard = template_match(tomo_hard_b, tpl_b["4v4r"], ROTS, mask_rib, FILT16, snapshots=SNAPS)
print(f"second search at SNR {SNR_HARD}: {time.time() - t0:.1f} s")
fig, ax = plt.subplots(figsize=(4.8, 3.7))
best_f1 = {}
for snr, sc, col in [(SNR_MAIN, score_rib, "tab:red"), (SNR_HARD, score_hard, "tab:purple")]:
    pk, _ = nms_peaks(sc, min_dist=R_BS["4v4r"] / 2)
    P_, R_, F_ = pr_curve(to_full_res(pk), gt_xyz[is_rib], gt_rad[is_rib], k_max=60)
    kb = int(np.argmax(F_))
    best_f1[snr] = F_[kb]
    ax.plot(R_, P_, "-o", ms=3, color=col, label=f"SNR {snr}: best F1 = {F_[kb]:.2f} (top {kb + 1} peaks)")
    ax.scatter(R_[kb], P_[kb], s=120, facecolors="none", edgecolors="k")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_xlim(0, 1.03); ax.set_ylim(0, 1.03)
ax.set_title("Ribosome template matching: precision–recall"); ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()
print(f"true number of ribosomes: {is_rib.sum()}; best F1 at SNR {SNR_MAIN}: {best_f1[SNR_MAIN]:.2f}, "
      f"at SNR {SNR_HARD}: {best_f1[SNR_HARD]:.2f}")
assert best_f1[SNR_MAIN] > 0.7, "ribosome template matching should work well at SNR 0.03"

🔍 **What just happened?** For a large, distinctive molecule at SNR 0.03, template matching ranks (nearly) all ribosomes
above everything else: the curve hugs the top-right corner. Imaging the same cell six times noisier (SNR 0.005) bends the
curve: to reach high recall you must accept false positives. Real data add further difficulties (thicker samples,
imperfect templates, CTF and alignment errors), which the next experiments probe.

### 2.5 How many rotations are enough?

The rotation set is the expensive part: run time grows linearly with the number of rotations. Too few rotations and some
ribosomes are never "seen" in a matching orientation. During both searches we stored the running best map after 1, 3,
10, 30, 100 and 300 rotations — let us score each.

In [ ]:
# ── 2.5 Recall/F1 as a function of the number of rotations tried ──
fig, ax = plt.subplots(figsize=(5.2, 3.3))
for snr, snaps, col in [(SNR_MAIN, snaps_rib, "tab:red"), (SNR_HARD, snaps_hard, "tab:purple")]:
    f1s = []
    for k in SNAPS:
        pk, _ = nms_peaks(snaps[k], min_dist=R_BS["4v4r"] / 2)
        f1s.append(pr_curve(to_full_res(pk), gt_xyz[is_rib], gt_rad[is_rib], k_max=40)[2].max())
    print(f"SNR {snr}: best F1 after " + ", ".join(f"{k} rot: {f:.2f}" for k, f in zip(SNAPS, f1s)))
    ax.semilogx(SNAPS, f1s, "-o", color=col, label=f"SNR {snr}")
ax.set_xlabel("number of template rotations tried (log scale)"); ax.set_ylabel("best F1")
ax.set_title("Rotation sampling vs picking quality"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At 2 nm voxels the ribosome's roundish outline matches fairly well even in one orientation
(F1 ≈ 0.8 with a single rotation), but it takes a few dozen to find (nearly) all of them. At the harder SNR something
surprising happens: **trying more rotations can make picking worse** (compare 100 and 300 rotations, and 1 and 3). Each voxel keeps the *maximum* of its scores over all rotations, and the maximum of many noisy numbers
creeps upward as you try more of them (a "multiple comparisons" effect, Module 04). Background voxels gain from this as
much as true particles once the right orientation is already covered — so the noise floor rises towards the true peaks.
Two lessons: at 2 nm voxels a coarse rotation set is enough *for detection* (estimating each particle's **orientation**
precisely is a separate, harder problem — Modules 12–13), and a template-matching score is a maximum over many trials,
whose chance level must be calibrated (Section 4). A uniform sampling at angular step $\Delta$ needs roughly $8\pi^2/\Delta^3$ rotations ($\Delta$ in radians):
≈ 550 at 30°, ≈ 4,400 at 15°, ≈ 35,000 at 7.5° — the cost explodes as you ask for finer angles.

### 2.6 Pitfall: look-alike molecules

Template matching answers "where does something *like this template* sit?" — not "where is *this molecule*?". The singly
capped proteasome (6UTJ) *contains* the 20S barrel (3J9I). What happens when we search for 3J9I?

In [ ]:
# ── 2.6 Search with the 20S proteasome template; inspect scores at every true particle ──
mask_20s = sphere_mask(16, support_radius(tpl_b["3j9i"], 0.2) + 0.5)
t0 = time.time()
score_20s, _, _ = template_match(tomo_b, tpl_b["3j9i"], ROTS, mask_20s, FILT16)
print(f"20S search done in {time.time() - t0:.1f} s")
local_best = ndimage.maximum_filter(score_20s, size=3)          # tolerate ±1 binned voxel
idx_b = np.clip(np.round((gt_xyz - 0.5) / 2).astype(int), 0, np.array(tomo_b.shape) - 1)
score_at_truth = local_best[tuple(idx_b.T)]
bg_peaks, bg_vals = nms_peaks(score_20s, min_dist=4)
far = np.min(np.linalg.norm(to_full_res(bg_peaks)[:, None] - gt_xyz[None], axis=-1), axis=1) > 20
fig, ax = plt.subplots(figsize=(6.2, 3.4))
groups = [score_at_truth[gt_cls == pid] for pid in CAST] + [bg_vals[far][:60]]
labels = [f"{NAMES[pid]}\n({pid.upper()})" for pid in CAST] + ["background\npeaks"]
cols = [MARK[pid][1] for pid in CAST] + ["0.5"]
for i, (g, col) in enumerate(zip(groups, cols)):
    ax.scatter(np.full(len(g), i) + rng.uniform(-0.15, 0.15, len(g)), g, color=col, s=14)
    ax.hlines(np.median(g), i - 0.3, i + 0.3, color="k")
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("20S-template score"); ax.set_title("Searching for the 20S proteasome (3J9I)")
plt.tight_layout(); plt.show()
pk20, _ = nms_peaks(score_20s, min_dist=R_BS["3j9i"] / 2)
is_20s = gt_cls == "3j9i"
_, _, F20 = pr_curve(to_full_res(pk20), gt_xyz[is_20s], gt_rad[is_20s], k_max=40)
print(f"median score: 20S {np.median(groups[1]):.2f} | capped {np.median(groups[2]):.2f} | "
      f"ribosome {np.median(groups[0]):.2f} | RuBisCO {np.median(groups[3]):.2f} | background {np.median(groups[4]):.2f}")
print(f"best F1 for finding 20S proteasomes only: {F20.max():.2f}")

🔍 **What just happened?** The 20S template scores the **capped** proteasomes almost as high as the real 20S ones, so a
list of "20S hits" is contaminated with capped proteasomes no matter which threshold you choose — the best $F_1$ is clearly
lower than for the ribosome. This is the fundamental limit of template matching: **it finds what resembles the template**.
Distinguishing compositional or conformational variants needs classification after picking (Modules 14–16), and finding
molecules for which no template exists needs template-free methods (Modules 15, 22).

✏️ **Your turn.** (a) In cell 2.2 change `N_ROT = 300` to `30` and re-run 2.2–2.4: how much faster, how much worse?
(b) In cell 1.1 set `SNR_MAIN = 0.01` and re-run Sections 1–2. Which molecule's picking breaks first?
<details><summary>Hint</summary>At SNR 0.01 the ribosome is still found reasonably well (large mass = more signal),
while the 20S/capped confusion gets worse because the small differences (the cap) are drowned first.</details>

---
## 3 · Difference of Gaussians: picking without a template

🧫 Template matching needs to know the molecule in advance. Often you do not — you just want *every* compact dense object
of roughly the right size, to classify later. The simplest such detector is a **blob detector**.

### 3.1 The difference-of-Gaussians (DoG) filter

Blur the tomogram with two Gaussians of slightly different widths and subtract:

$$I_{\text{DoG}}=I*G_{\sigma_1}-I*G_{\sigma_2},\qquad \sigma_2=k\,\sigma_1,\quad k=1.1,\qquad
G_\sigma(r)\propto e^{-r^2/2\sigma^2}.$$

*In words:* the narrow blur keeps blobs of size ≈ σ₁ and smaller; the slightly wider blur removes a bit more; their
difference keeps a **narrow band of sizes** around σ₁ — a band-pass filter (Module 03). Blobs of the matching size light
up as local maxima, smaller details (noise) and larger structures (gradients in ice thickness) are suppressed.
$k=1.1$ is the value used by Pei *et al.* (2016) and by AITom's `aitom/pick/dog` (with $k\to1$, DoG becomes the
Laplacian-of-Gaussian blob detector).

🧮 **Which σ for which particle?** For a uniform solid sphere of radius $r$, the DoG response at its centre is
proportional to $u^3e^{-u^2/2}$ with $u=r/\sigma$ (for $k\to1$), which peaks at $u=\sqrt3$. So the detector is tuned to
spheres of radius $r\approx\sqrt3\,\sigma_1\approx1.7\,\sigma_1$. The next cell checks this numerically.

In [ ]:
# ── 3.1 DoG response at the centre of a solid sphere, as a function of sphere radius ──
def dog_center_response(r, s1, k=1.1):
    """Exact DoG value at the centre of a uniform sphere of radius r: P(|X|≤r) for a 3D Gaussian X of width σ
    equals the χ (3 d.o.f.) CDF at r/σ."""
    return stats.chi(3).cdf(r / s1) - stats.chi(3).cdf(r / (k * s1))


radii = np.linspace(0.5, 25, 200)
fig, ax = plt.subplots(figsize=(6.0, 3.3))
for s1, col in zip([3, 5, 7, 9], plt.cm.viridis(np.linspace(0, 0.9, 4))):
    resp = dog_center_response(radii, s1)
    ax.plot(radii, resp / resp.max(), color=col, label=f"σ₁ = {s1} nm (peak at r = {radii[np.argmax(resp)]:.1f} nm)")
for pid in CAST:
    ax.axvline(R_BS[pid] * 0.75, color=MARK[pid][1], ls=":", lw=1)
    ax.text(R_BS[pid] * 0.75 + 0.2, 0.02, pid.upper(), color=MARK[pid][1], fontsize=7, rotation=90, va="bottom")
ax.set_xlabel("sphere radius r (nm)"); ax.set_ylabel("normalized DoG response at centre")
ax.set_title("DoG is a size-tuned detector: best for r ≈ √3·σ₁"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
r_peak = radii[np.argmax(dog_center_response(radii, 5))]
print(f"peak radius / σ₁ = {r_peak / 5:.2f} for k = 1.1 (the k → 1 limit is √3 = {np.sqrt(3):.2f})")

🔍 The dotted lines mark ~¾ of each molecule's bounding radius (a rough "effective" radius, since molecules are not solid
spheres). Each σ prefers one size band: σ₁ ≈ 3–5 nm suits RuBisCO and the proteasomes, σ₁ ≈ 7–9 nm the ribosome.

### 3.2 The full Pei *et al.* recipe

1. Compute $I_{\text{DoG}}$ with $\sigma_2=1.1\sigma_1$ and find all **local maxima** (3 × 3 × 3 neighbourhood).
2. Let $M$ and $m$ be the largest and smallest local-maximum values. Keep peaks above
   $$T=m+t\,\frac{M-m}{K},\qquad K=20,$$
   *in words:* divide the range of peak heights into 20 equal bins and keep the peaks in the top $20-t$ of them; $t$ is
   the integer "threshold level" the user tunes (the paper's best was $t=3$ with $\sigma=7$ nm).
3. Sort the survivors and remove any peak closer than a minimum distance to a stronger one (we use $2\sigma_1$, as AITom's
   `particle_picking_dog__filter.py` does).
4. Score with the TP rule $S\le0.5$.

In [ ]:
# ── 3.2 DoG picking exactly as in Pei et al. (2016) ──
def dog_filter(vol, s1, k=1.1):
    """Difference of Gaussians I*G(σ1) − I*G(k·σ1)."""
    return ndimage.gaussian_filter(vol, s1) - ndimage.gaussian_filter(vol, k * s1)


def dog_candidates(vol, s1):
    """All local maxima of the DoG map, strongest first: (positions [n, 3], values)."""
    D = dog_filter(vol, s1)
    pk = np.argwhere(D == ndimage.maximum_filter(D, size=3))
    vals = D[tuple(pk.T)]
    order = np.argsort(-vals)
    return pk[order], vals[order]


def remove_close(peaks, min_dist):
    """Greedy suppression of peaks closer than min_dist to a stronger (earlier) peak; returns a boolean keep mask."""
    keep = np.ones(len(peaks), bool)
    tree = cKDTree(peaks)
    for i in range(len(peaks)):
        if keep[i]:
            for j in tree.query_ball_point(peaks[i], min_dist):
                if j > i:
                    keep[j] = False
    return keep


def dog_pick(cands, vals, s1, t, K=20, max_peaks=6000):
    """Threshold rule T = m + t (M − m)/K, then distance filtering. Returns (picks, threshold T)."""
    M, m = vals[0], vals[-1]
    T = m + t * (M - m) / K
    sel = vals >= T
    if sel.sum() > max_peaks:             # far too many peaks: this threshold is useless anyway
        return np.zeros((0, 3)), T
    p = cands[sel]
    return p[remove_close(p, 2 * s1)], T


def dog_sweep(vol, s1, true_zyx, true_radius, t_values=range(0, 20)):
    """Run DoG once and evaluate every threshold level t. Returns a list of (t, metrics, found)."""
    cands, vals = dog_candidates(vol, s1)
    out = []
    for t in t_values:
        picks, _ = dog_pick(cands, vals, s1, t)
        if len(picks):
            m, found, _ = match_pei(picks, true_zyx, true_radius)
            out.append((t, m, found, picks))
    return out


t0 = time.time()
sweep5 = dog_sweep(tomo, 5.0, gt_xyz, gt_rad)
best5 = max(sweep5, key=lambda r: r[1]["f1"])
print(f"DoG σ₁ = 5 nm on the full 1-nm tomogram ({time.time() - t0:.1f} s)")
for t, m, _, _ in sweep5[::2]:
    print(f"  t = {t:2d}: {m['n_picks']:5d} picks  P = {m['precision']:.2f}  R = {m['recall']:.2f}  F1 = {m['f1']:.2f}")
print(f"best: t = {best5[0]}, F1 = {best5[1]['f1']:.2f}")

In [ ]:
# ── 3.3 Where are the DoG picks, and which are wrong? ──
t_b, _, _, picks_b = best5
m_b, found_b, found_pick = match_pei(picks_b, gt_xyz, gt_rad)       # found_pick: which picks hit a particle
z_ves = 32                                                         # the plane through the vesicle centre
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.4))
axes[0].imshow(dog_filter(tomo, 5.0)[z_ves], cmap="viridis"); axes[0].set_title(f"DoG map (σ₁ = 5 nm), z = {z_ves} nm")
axes[1].imshow(ndimage.gaussian_filter(tomo[z_ves], 1)); axes[1].set_title(f"picks at t = {t_b}: green = hit, magenta = false")
for ax in axes:
    draw_truth(ax, z_ves, s=90)
sel = np.abs(picks_b[:, 0] - z_ves) <= 6
axes[1].scatter(picks_b[sel & found_pick, 2], picks_b[sel & found_pick, 1], marker="x", color="lime", s=40)
axes[1].scatter(picks_b[sel & ~found_pick, 2], picks_b[sel & ~found_pick, 1], marker="x", color="magenta", s=40)
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()
miss = {NAMES[pid]: int(np.sum(~found_b & (gt_cls == pid))) for pid in CAST}
print("missed particles per class:", miss)
ves_c, ves_r = np.array([32, 0.78 * 160, 0.22 * 160]), 0.16 * 160          # vesicle geometry used by make_crowded_cell
on_membrane = np.abs(np.linalg.norm(picks_b - ves_c, axis=1) - ves_r) <= 6
grid = np.argwhere(np.ones((16, 40, 40), bool)) * 4 + 2                        # coarse grid over the whole box
shell_frac = np.mean(np.abs(np.linalg.norm(grid - ves_c, axis=1) - ves_r) <= 6)
print(f"false picks: {np.sum(~found_pick)} of {len(picks_b)}; {np.sum(~found_pick & on_membrane)} of them lie within 6 nm "
      f"of the membrane, a shell that fills only {100 * shell_frac:.0f} % of the volume")

🔍 **What just happened?** DoG needed **no template** and still found most particles of all four kinds, in about a second.
A large share of its false positives (magenta) sit on the **membrane vesicle** — a membrane is dense and bright at the DoG
scale, but it is not a particle — and occasionally a large ribosome is split into two picks. DoG cannot tell *what* it found; that is
the job of classification (Modules 14–16, 22).

### 3.4 Scale matters: per-class recall vs σ₁

In [ ]:
# ── 3.4 Per-class recall at the best threshold, for several DoG scales ──
sigmas = [2, 3, 4, 5, 6, 7, 9]
per_class, bestF = {pid: [] for pid in CAST}, []
t0 = time.time()
for s1 in sigmas:
    sw = dog_sweep(tomo, s1, gt_xyz, gt_rad, t_values=range(2, 16))
    t, m, found, _ = max(sw, key=lambda r: r[1]["f1"])
    bestF.append(m["f1"])
    for pid in CAST:
        per_class[pid].append(found[gt_cls == pid].mean())
print(f"σ sweep done in {time.time() - t0:.1f} s")
fig, ax = plt.subplots(figsize=(6.0, 3.4))
for pid in CAST:
    ax.plot(sigmas, per_class[pid], "-o", ms=4, color=MARK[pid][1], label=f"recall: {NAMES[pid]}")
ax.plot(sigmas, bestF, "k--", lw=2, label="overall best F1")
ax.set_xlabel("σ₁ (nm)"); ax.set_ylabel("recall / F1"); ax.set_ylim(0, 1.05)
ax.set_title("DoG scale vs particle size"); ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()
print("best overall σ₁ =", sigmas[int(np.argmax(bestF))], "nm")

🔍 **What just happened?** Small σ₁ finds the small RuBisCO but **fragments** ribosomes into several sub-blobs (each counts
once as a hit and the rest as false positives); large σ₁ merges or misses the small molecules. The overall optimum is a
compromise for this particular mixture of sizes — exactly why Pei *et al.* reported that **combining σ = 5, 7 and 9 nm
improved F for all size groups**.

✏️ **Your turn.** Implement multi-scale DoG: pool the picks of σ₁ = 3, 5 and 8 (use each scale's best $t$), remove
duplicates closer than 6 nm with `remove_close`, and score the union with `match_pei`. Does it beat the best single scale?

### 3.5 Crowding: the paper's key finding

Pei *et al.* simulated cytoplasm at 11 %, 26 % and 44 % volume occupancy and found that the **maximum recall of DoG fell
with crowding**. We repeat the experiment with our packer at four crowding levels (no membrane, SNR 0.01), using both σ₁ = 5
nm and the paper's σ₁ = 7 nm. Occupancy here = total bounding-sphere volume / box volume, the paper's definition.

In [ ]:
# ── 3.5 DoG vs crowding ──
crowd_rows = []
t0 = time.time()
for f in [0.5, 1.0, 2.0, 3.0]:
    counts_f = {pid: int(round(n * f)) for pid, n in COUNTS.items()}
    cl, parts_f = make_crowded_cell(zoo32, counts_f, shape=(64, 160, 160), rng=np.random.default_rng(SEED + 7),
                                    membrane=False)
    xyz_f = np.array([p["center_zyx"] for p in parts_f])
    rad_f = np.array([R_BS[p["cls"]] for p in parts_f])
    occ_f = np.sum(4 / 3 * np.pi * rad_f**3) / cl.size
    tomo_f = simulate_tomogram(cl, np.random.default_rng(SEED + 8), snr=0.01)
    for s1 in (5.0, 7.0):
        sw = dog_sweep(tomo_f, s1, xyz_f, rad_f, t_values=range(2, 16))
        crowd_rows.append(dict(occ=occ_f, n=len(parts_f), s1=s1,
                               maxR=max(r[1]["recall"] for r in sw), maxF=max(r[1]["f1"] for r in sw)))
    print(f"occupancy {100 * occ_f:4.1f} % ({len(parts_f):3d} particles): "
          + " | ".join(f"σ₁={r['s1']:.0f}: max recall {r['maxR']:.2f}, max F1 {r['maxF']:.2f}" for r in crowd_rows[-2:]))
print(f"crowding experiment: {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2))
for s1, ls in [(5.0, "-o"), (7.0, "--s")]:
    rr = [r for r in crowd_rows if r["s1"] == s1]
    axes[0].plot([100 * r["occ"] for r in rr], [r["maxR"] for r in rr], ls, label=f"σ₁ = {s1:.0f} nm")
    axes[1].plot([100 * r["occ"] for r in rr], [r["maxF"] for r in rr], ls, label=f"σ₁ = {s1:.0f} nm")
for ax, lab in zip(axes, ["maximum recall", "maximum F1"]):
    ax.set_xlabel("crowding: bounding-sphere occupancy (%)"); ax.set_ylabel(lab); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
axes[0].set_title("Recall falls with crowding"); axes[1].set_title("…and so does F1")
plt.tight_layout(); plt.show()

🔍 **What just happened?** As the box fills up, neighbouring particles start to touch at the DoG scale: two blobs merge
into one peak, or a large neighbour's peak swallows a small one, and recall drops — most strongly at the larger σ₁ = 7 nm,
whose wider Gaussians mix neighbours more. This is the qualitative trend of Pei *et al.*; the exact numbers differ
(different molecules, packing, SNR definition). Note that our rejection-sampling packer cannot reach the paper's 44 %
occupancy — dense packing needs the physics-based packing of Module 09.

### 3.6 Noise

In [ ]:
# ── 3.6 DoG vs SNR (same cell as Section 1, σ₁ = 5 nm) ──
snrs = [1.0, 0.1, 0.03, 0.01, 0.003]
noise_rows = []
for snr in snrs:
    tomo_s = simulate_tomogram(clean, np.random.default_rng(SEED + 1), snr=snr)
    sw = dog_sweep(tomo_s, 5.0, gt_xyz, gt_rad, t_values=range(2, 16))
    _, m, _, _ = max(sw, key=lambda r: r[1]["f1"])
    noise_rows.append(m)
    print(f"SNR {snr:6.3f}: best F1 {m['f1']:.2f} (P {m['precision']:.2f}, R {m['recall']:.2f})")
fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.semilogx(snrs, [m["f1"] for m in noise_rows], "-o", color="k", label="F1")
ax.semilogx(snrs, [m["precision"] for m in noise_rows], ":", color="tab:blue", label="precision")
ax.semilogx(snrs, [m["recall"] for m in noise_rows], ":", color="tab:orange", label="recall")
ax.invert_xaxis(); ax.set_xlabel("SNR (log scale; noisier →)"); ax.set_ylim(0, 1.05)
ax.set_title("DoG picking vs noise"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** DoG is robust down to moderate noise because the Gaussian blur averages many voxels; at very
low SNR, noise maxima become as tall as particle maxima and $F_1$ collapses — the same shape as the paper's curve (stable
down to SNR 10 in *their* units, dropping at SNR 1). ⚠️ SNR numbers are **not comparable across papers**: Pei *et al.*
add noise before and after the CTF and define SNR in their own way; we use var(signal)/var(noise) of the final tomogram.

### 📄 Paper connection — Pei, Xu, Frazier & Alber (2016)

**Paper:** Pei L, Xu M, Frazier Z, Alber F. *Simulating cryo electron tomograms of crowded cell cytoplasm for assessment of
automated particle picking.* BMC Bioinformatics 17:405 (2016). [doi:10.1186/s12859-016-1283-3](https://doi.org/10.1186/s12859-016-1283-3)
(Note: AITom's publication list uses a paraphrased title, "…Crowded Mixtures of Macromolecular Complexes and Assessment of
Particle Picking"; the published title is the one above.)

- **Question it answers:** how well do automatic pickers work in *crowded* cytoplasm, where the truth is unknown?
  Answer it with realistic simulations whose ground truth is known.
- **What the paper actually does:** converts 21 abundant complexes to density maps (4 nm resolution, 1 nm voxels),
  represents each by its minimum bounding sphere (contour at 0.2 × max density), packs thousands of spheres without overlap
  with IMP molecular dynamics plus **simulated annealing**, simulates tilt series (±60°, 2° steps, 300 kV, $C_s$ = 2 mm,
  defocus −4 µm, CTF + MTF, noise), reconstructs by back-projection, then picks with **DoG** ($\sigma_2=1.1\sigma_1$,
  threshold $T=m+t(M-m)/20$, minimum-distance merging) and scores with $S=|x_p-C_g|/R_g\le0.5$, one peak per particle.
- **What our toy keeps:** the DoG recipe, the threshold rule, the TP rule, the bounding-sphere occupancy measure, sweeps over
  σ, crowding and SNR.
- **What our toy leaves out:** MD/annealing packing (we use rejection sampling, which cannot reach 44 %), tilt-series
  simulation (we use a direct Fourier wedge + isotropic CTF), the paper's 21 complexes and its SNR definition.
- **Result reported:** the authors report a best $F=0.831$ (precision 96.9 %, recall 72.7 %) at σ = 7 nm, $t=3$, 11 %
  crowding and SNR 50; maximum recall falling from 78.7 % to 63.2 % to 52.0 % (and maximum $F$ from 0.831 to 0.717 to 0.637)
  at 11 %, 26 % and 44 % crowding; $F$ near 0.8 down to SNR 10 and 0.546 at SNR 1; and that combining σ = 5, 7, 9 nm
  improves $F$ for all size groups.
- **Code:** AITom `aitom/pick/dog/` (DoG peaks + distance filter; tutorial `doc/tutorials/008_particle_picking.py`) and
  `aitom/simulation/tomogram/single_bounding_sphere/` (packing + simulation).

---
## 4 · Is a match real? Template-matching scores need a null distribution

🧫 Suppose template matching tells you that a subtomogram matches the ribosome better than any other template in your
library. Is it a ribosome? Not necessarily: **the highest score is always *some* template**, even for a box of pure noise or
for a molecule not in your library. A score of 0.3 means nothing until you know how high scores get *by chance*.

**Wang *et al.* (2018)** framed this as a hypothesis test. For a subtomogram $P$ whose best-scoring library complex is $C_I$:

$$H_0:\;P\ \text{is not a copy of}\ C_I\qquad\text{vs.}\qquad H_A:\;P\ \text{is a copy of}\ C_I .$$

To get the distribution of scores *under $H_0$* you need many plausible "other molecules" — far more than the known
structures. Their answer: **generate pseudo-macromolecules** with a generative model trained on real density maps, keep
only those that are *far* from $C_I$, align $P$ against each of $B$ such pseudo-templates, and compute a Monte Carlo
p-value

$$\hat p=\frac1B\sum_{b=1}^{B}\mathbb 1\!\left[c\big(P,C_I\big)\le c\big(P,C_0^{(b)}\big)\right].$$

*In words:* the p-value is the fraction of random look-alike-but-different molecules that fit $P$ at least as well as the
claimed match. A match is accepted only if $\hat p\le0.01$.

Our toy keeps this logic but replaces the paper's 3D Wasserstein GAN with a cheap generator: random assemblies of 3–12
globular "domains" (Gaussian blobs) of protein-like size.

In [ ]:
# ── 4.1 Query subtomograms: real particles and empty (noise-only) boxes, binned to 20³ at 2 nm ──
def extract_box(vol, center_zyx, size):
    """Cut a size³ box centred on a (rounded) position, zero-padding outside the volume."""
    c = np.round(center_zyx).astype(int)
    h = size // 2
    pad = np.pad(vol, h, mode="constant")
    return pad[c[0]:c[0] + size, c[1]:c[1] + size, c[2]:c[2] + size]


N_PER_CLASS, N_EMPTY = 8, 10
queries, q_type = [], []
for pid in ["4v4r", "3j9i", "1bxn"]:
    for p in [p for p in particles if p["cls"] == pid][:N_PER_CLASS]:
        queries.append(bin2(extract_box(tomo, p["center_zyx"], 40)))
        q_type.append(pid)
noise_sd = tomo.std() / np.sqrt(1 + SNR_MAIN)          # total variance = signal + noise variance
rng_q = np.random.default_rng(SEED + 20)
for _ in range(N_EMPTY):
    queries.append(bin2(rng_q.normal(0, noise_sd, (40, 40, 40)).astype(np.float32)))
    q_type.append("empty")
queries, q_type = np.array(queries), np.array(q_type)
print("queries:", queries.shape, {t: int(np.sum(q_type == t)) for t in ["4v4r", "3j9i", "1bxn", "empty"]})

In [ ]:
# ── 4.2 A cheap generator of pseudo-macromolecules ──
def pseudo_structure(rng, n=20):
    """Random 'molecule': 3–12 Gaussian domains (σ 1–2.2 voxels) inside a sphere of radius 3–8 voxels (2 nm voxels)."""
    g = np.arange(n) - (n - 1) / 2
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r_max, k = rng.uniform(3, 8), rng.integers(3, 13)
    v = np.zeros((n, n, n))
    for _ in range(k):
        d = rng.normal(size=3)
        c = d / np.linalg.norm(d) * r_max * rng.uniform(0, 1) ** (1 / 3)   # uniform point in the sphere
        s = rng.uniform(1.0, 2.2)
        v += rng.uniform(0.5, 1) * np.exp(-((z - c[0])**2 + (y - c[1])**2 + (x - c[2])**2) / (2 * s * s))
    v -= v.min()
    return (v / v.max()).astype(np.float32)


B = 150
pseudo = np.array([pseudo_structure(np.random.default_rng(5000 + b)) for b in range(B)])
real20 = np.array([bin2(zoo40[pid]) for pid in CAST])            # 20³ at 2 nm
fig, axes = plt.subplots(2, 6, figsize=(9.0, 3.4))
for ax, v, lab in zip(axes[0], list(real20) + list(pseudo[:2]), [pid.upper() for pid in CAST] + ["pseudo 1", "pseudo 2"]):
    ax.imshow(v.sum(0)); ax.set_title(lab, fontsize=8); ax.axis("off")
for i, ax in enumerate(axes[1]):
    ax.imshow(pseudo[2 + i].sum(0)); ax.set_title(f"pseudo {3 + i}", fontsize=8); ax.axis("off")
fig.suptitle("Real templates (top left) and pseudo-macromolecules: projections, 40 × 40 nm boxes", fontsize=9)
plt.tight_layout(); plt.show()

🔍 The pseudo-structures are not real proteins — they only need to be **plausible alternatives**: compact, of protein-like
size and texture. The paper's GAN made them far more realistic (it learned shapes from 15 real maps).

### 4.3 The alignment score

To score a query $P$ against a template $T$ we need the best match over orientations. We use the same 150-rotation set for
every template and compute a **wedge- and CTF-aware correlation** in Fourier space: rotating the query by $R^{-1}$ also
rotates its missing wedge, so for each rotation we compare only Fourier coefficients the query actually measured (and only
up to 0.3 cycles/voxel ≈ 1/(67 Å), where the signal lives):

$$c(P,T)=\max_R\frac{\sum_{\xi}\widehat{R^{-1}P}(\xi)\,\overline{\hat T(\xi)}\,w_R(\xi)^2}
{\sqrt{\sum_\xi|\widehat{R^{-1}P}(\xi)|^2w_R(\xi)^2\;\sum_\xi|\hat T(\xi)|^2w_R(\xi)^2}},\qquad w_R(\xi)=\text{CTF}(\xi)\,W(R\xi).$$

*In words:* a Pearson correlation computed only over the frequencies that both the query and the (imaged) template can share
— the "constrained correlation" idea you will study properly in Module 12. With all templates stacked, one matrix
multiplication scores every (rotation, template) pair at once. (For simplicity we do not search translations: queries are
centred on the particle.)

In [ ]:
# ── 4.3 Fourier-space scoring of every query against every template ──
n20 = 20
k20 = freq_grid((n20,) * 3)
kr20 = np.sqrt(sum(kk**2 for kk in k20))
BAND = (kr20 > 0) & (kr20 <= 0.3)                                 # frequencies used (DC excluded)
k_xyz = np.stack([k20[2][BAND], k20[1][BAND], k20[0][BAND]])      # (3, n_freq) in (x, y, z) order
ctf_band = -ctf_2d((n20,) * 3, 20.0, defocus_um=5.0)[BAND]
ROTS_Q = ROTS[:150]
W_R = np.array([(np.abs((R @ k_xyz)[2]) <= np.abs((R @ k_xyz)[0]) * np.tan(np.radians(60)) + 1e-9)
                for R in ROTS_Q], np.float32)                     # wedge of R⁻¹P sampled at ξ is W(Rξ)
FILT2 = (W_R * ctf_band[None]) ** 2                               # w_R(ξ)², shape (n_rot, n_freq)


def score_queries(queries, templates):
    """c(P, T) for all queries × templates (max over the rotation set). Returns array [n_query, n_template]."""
    FT = np.fft.fftn(templates, axes=(1, 2, 3))[:, BAND]          # (n_tpl, n_freq)
    den_T = np.sqrt((np.abs(FT) ** 2) @ FILT2.T)                  # (n_tpl, n_rot)
    out = np.zeros((len(queries), len(templates)))
    for qi, q in enumerate(queries):
        FQ = np.fft.fftn(np.stack([rotate_volume(q, R.T) for R in ROTS_Q]), axes=(1, 2, 3))[:, BAND]
        num = np.real((FQ * FILT2) @ np.conj(FT).T)               # (n_rot, n_tpl)
        den_Q = np.sqrt(np.sum(np.abs(FQ) ** 2 * FILT2, axis=1))
        out[qi] = (num / (den_Q[:, None] * den_T.T)).max(axis=0)
    return out


t0 = time.time()
all_templates = np.concatenate([real20, pseudo])                  # first 4 = real library, then B pseudo
SC = score_queries(queries, all_templates)
print(f"scored {len(queries)} queries × {len(all_templates)} templates × {len(ROTS_Q)} rotations "
      f"in {time.time() - t0:.1f} s")

In [ ]:
# ── 4.4 Rejection region, best template and Monte Carlo p-values ──
TAU = 0.8
sph = sphere_mask(n20, 9.5) > 0


def unit_rows(X):
    """Zero-mean, unit-norm rows."""
    X = X - X.mean(axis=1, keepdims=True)
    return X / np.linalg.norm(X, axis=1, keepdims=True)


Ps = unit_rows(pseudo[:, sph])
sim = np.zeros((len(CAST), B))                                    # max over rotations of corr(clean real, clean pseudo)
for i, v in enumerate(real20):
    Rv = unit_rows(np.stack([rotate_volume(v, R)[sph] for R in ROTS_Q]))
    sim[i] = (Rv @ Ps.T).max(axis=0)
print("pseudo-structures excluded from each null (too similar, sim ≥ 0.8):",
      {pid.upper(): int(np.sum(sim[i] >= TAU)) for i, pid in enumerate(CAST)})

results = []
for qi in range(len(queries)):
    best = int(np.argmax(SC[qi, :4]))
    null = SC[qi, 4:][sim[best] < TAU]
    p_hat = float(np.mean(null >= SC[qi, best]))
    results.append(dict(type=q_type[qi], best=CAST[best], score=SC[qi, best], p=p_hat, null=null))
print(f"{'query':>8s} | {'best template':>13s} | {'accepted by highest-score rule':>30s} | {'p ≤ 0.01':>8s}")
for t in ["4v4r", "3j9i", "1bxn", "empty"]:
    rr = [r for r in results if r["type"] == t]
    bests = ", ".join(f"{b.upper()}×{sum(r['best'] == b for r in rr)}" for b in CAST if any(r["best"] == b for r in rr))
    print(f"{t:>8s} | {bests:>13s} | {len(rr):>27d}/{len(rr)} | {sum(r['p'] <= 0.01 for r in rr):>5d}/{len(rr)}")

In [ ]:
# ── 4.5 Two null distributions side by side ──
r_real = next(r for r in results if r["type"] == "4v4r")
r_empty = max([r for r in results if r["type"] == "empty"], key=lambda r: r["p"])
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.2))
for ax, r, title in zip(axes, [r_real, r_empty], ["a ribosome subtomogram", "an EMPTY (noise-only) box"]):
    ax.hist(r["null"], bins=25, color="0.7", label=f"null: {len(r['null'])} pseudo-structures")
    ax.axvline(r["score"], color="tab:red", lw=2, label=f"best real template ({r['best'].upper()})")
    ax.set_title(f"{title}: p̂ = {r['p']:.2f}"); ax.set_xlabel("alignment score c(P, T)")
    ax.legend(fontsize=7)
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()
emp = [r for r in results if r["type"] == "empty"]
print(f"empty boxes: the highest-score rule 'identifies' all {len(emp)} of them; "
      f"the p-value test rejects {sum(r['p'] > 0.01 for r in emp)} of {len(emp)}")

🔍 **What just happened?** For a real ribosome, the ribosome template beats every pseudo-structure: $\hat p\approx0$. For a
box containing **only noise**, the best real template still gets a score — the highest-score rule would happily
report "ribosome" or "RuBisCO" — but a good fraction of random pseudo-structures fit the noise just as well, so $\hat p$ is
large and the claim is rejected. Notice also that the empty boxes' best scores are an order of magnitude lower than real
matches; but *how* low is "too low" depends on box size, SNR, mask and template, which is exactly why a calibrated null is
needed. (A few empty boxes may still pass — with 10 boxes and only ~100–150 pseudo-structures, the toy test is coarse.)

⚠️ **Template bias.** A related trap: if you *average* noise boxes after aligning them to a template, the average looks like
the template ("Einstein from noise"). Module 14 shows it.

### 📄 Paper connection — Wang, Zeng, Liang, Huo, Xing & Xu (2018)

**Paper:** Wang KW, Zeng X, Liang X, Huo Z, Xing EP, Xu M. *Image-derived generative modeling of pseudo-macromolecular
structures – towards the statistical assessment of Electron CryoTomography template matching.* British Machine Vision
Conference (BMVC) 2018 (venue per the lab bibliography); [arXiv:1805.04634](https://arxiv.org/abs/1805.04634).

- **Question it answers:** is a template-matching hit statistically credible, or would a random other molecule fit as well?
- **What the paper actually does:** trains a **3D Wasserstein GAN with gradient penalty** (λ = 10; 3D-DCGAN-style networks
  with four stride-2 convolutions, 100-dimensional latent code, minibatch discrimination) on rotated copies of 15 PDB-derived
  maps (64³ at 0.6 nm); samples ≈ 1,000 pseudo-structures **far from** the top-scoring complex $C_I$ (a rejection region
  built from an inverse generator, a kernel density estimate and a Bayes classifier); uses their alignment scores with $P$
  as the null distribution and computes $\hat p$; a match is claimed if $\hat p\le0.01$.
- **What our toy keeps:** the hypotheses, the Monte Carlo p-value, the "far from $C_I$" exclusion (a simple correlation
  threshold), and the comparison with the highest-score rule on empty boxes.
- **What our toy leaves out:** the GAN (we use random blob assemblies), the learned rejection region, translational search,
  and the scale (we use 150 pseudo-structures and 150 rotations).
- **Result reported:** the authors report 12 of 15 simulated tests succeeding; 264 of 376 (70.2 %) experimental ribosome
  subtomograms from EMPIAR-10045 (yeast 80S, PDB 4V7R) passing, including 92 of the top 100 by alignment score; and, on 20
  empty subtomograms, preventing 40 % of the false positives of the highest-score rule.
- **Code:** none located in AITom.

---
## 5 · Saliency without templates: supervoxels + robust PCA

🧫 Sometimes you do not even want "blobs of size σ" — you want to know **which parts of a huge tomogram are interesting at
all**, so that later, expensive analyses can skip the boring 90 %. Zhou *et al.* (2018) treated this as **saliency
detection** (a computer-vision term for "what stands out"):

1. smooth the tomogram and over-segment it into **supervoxels** — small, compact, homogeneous regions (3D SLIC);
2. describe each supervoxel by a feature vector (in the paper: 24 mean 3D-Gabor responses + a 6-bin density histogram);
3. stack the features into a matrix $F$ (one row per supervoxel) and split it with **robust PCA** (principal component
   pursuit)

$$\min_{L,S}\;\lVert L\rVert_*+\beta\lVert S\rVert_1\quad\text{s.t.}\quad F=L+S ;$$

*in words:* explain the feature matrix as a **low-rank** part $L$ (the background: most supervoxels look alike, so their
features lie in a few directions; $\lVert L\rVert_*$, the sum of singular values, rewards few directions) plus a **sparse**
part $S$ (a few supervoxels deviate strongly; $\lVert S\rVert_1$, the sum of absolute values, rewards few non-zeros).
4. The saliency of a supervoxel is the mean of its row of $S$.

We add a smooth **ice-thickness gradient** to the tomogram — real tomograms are never evenly bright — to see why simple
intensity is not enough. Our features: mean intensity, two DoG responses, oriented second derivatives (a cheap stand-in for
Gabor filters) and a 6-bin histogram.

In [ ]:
# ── 5.1 Supervoxels and features on the binned tomogram ──
from skimage.segmentation import slic
from sklearn.metrics import roc_auc_score

V = tomo_b.copy()
zz, yy, xx = np.meshgrid(*[np.linspace(-1, 1, s) for s in V.shape], indexing="ij")
V += 1.0 * V.std() * (xx + 0.5 * np.sin(3 * yy))                  # uneven "ice thickness" background
truth_b = bin2((clean > 0.15 * clean.max()).astype(np.float32)) > 0.5   # voxels inside particles/membrane
t0 = time.time()
Vs = ndimage.gaussian_filter(V, 0.9)                                # the paper smooths with σ = 1.8 voxels at full size
labels = slic(Vs, n_segments=800, compactness=0.05, channel_axis=None, start_label=0)
n_sv = labels.max() + 1
idx = np.arange(n_sv)
feats = [Vs, dog_filter(V, 1.5), dog_filter(V, 3.0)]
for s in (1.0, 2.0):
    for ax_ in range(3):
        order = [0, 0, 0]; order[ax_] = 2
        feats.append(-ndimage.gaussian_filter(V, s, order=order))  # "ridge/blob" response along one axis
F = np.stack([ndimage.mean(f, labels, idx) for f in feats], axis=1)
edges = np.linspace(Vs.min(), Vs.max(), 7)
counts_sv = np.bincount(labels.ravel(), minlength=n_sv)
H = np.stack([np.bincount(labels.ravel(), weights=((Vs >= edges[i]) & (Vs <= edges[i + 1])).ravel().astype(float),
                          minlength=n_sv) for i in range(6)], axis=1) / counts_sv[:, None]
F = np.concatenate([F, H], axis=1)
F = (F - F.mean(0)) / (F.std(0) + 1e-12)                            # standardize each feature
truth_sv = ndimage.mean(truth_b.astype(float), labels, idx) > 0.5  # supervoxel mostly inside a structure
print(f"{n_sv} supervoxels (~{counts_sv.mean():.0f} voxels each), feature matrix {F.shape}, "
      f"{100 * truth_sv.mean():.0f} % of supervoxels are 'structure'; {time.time() - t0:.1f} s")

In [ ]:
# ── 5.2 Robust PCA (principal component pursuit, inexact augmented Lagrangian) ──
def robust_pca(M, lam=None, tol=1e-7, max_iter=500):
    """Split M = L + S with L low-rank and S sparse: min ||L||_* + lam ||S||_1 (Candès et al. 2011 / Lin et al. 2010)."""
    lam = lam or 1 / np.sqrt(max(M.shape))
    norm2 = np.linalg.norm(M, 2)
    Y = M / max(norm2, np.abs(M).max() / lam)                        # dual variable
    mu, rho = 1.25 / norm2, 1.5
    S = np.zeros_like(M)
    for it in range(max_iter):
        U, sv, Vt = np.linalg.svd(M - S + Y / mu, full_matrices=False)
        L = (U * np.maximum(sv - 1 / mu, 0)) @ Vt                    # singular-value soft-thresholding
        T = M - L + Y / mu
        S = np.sign(T) * np.maximum(np.abs(T) - lam / mu, 0)         # element-wise soft-thresholding
        Z = M - L - S
        Y, mu = Y + mu * Z, min(mu * rho, 1e7)
        if np.linalg.norm(Z) / np.linalg.norm(M) < tol:
            break
    return L, S, it + 1


L_, S_, n_it = robust_pca(F)
saliency = S_.mean(axis=1)
U3, s3, Vt3 = np.linalg.svd(F, full_matrices=False)
pca_resid = F - (U3[:, :3] * s3[:3]) @ Vt3[:3]                      # what ordinary PCA (rank 3) cannot explain
aucs = {"raw intensity\n(mean of supervoxel)": roc_auc_score(truth_sv, F[:, 0]),
        "ordinary PCA\nresidual (rank 3)": roc_auc_score(truth_sv, pca_resid.mean(1)),
        "robust PCA\nsparse part S": roc_auc_score(truth_sv, saliency)}
print(f"RPCA converged in {n_it} iterations; rank(L) = {np.linalg.matrix_rank(L_, tol=1e-6)}, "
      f"{100 * np.mean(np.abs(S_) > 1e-6):.0f} % of S entries non-zero")
for k, v in aucs.items():
    print(f"  ROC AUC {k.replace(chr(10), ' '):38s} = {v:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.3), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
axes[0].imshow(V[zb]); axes[0].set_title("binned tomogram + ice gradient")
axes[1].imshow(saliency[labels][zb], cmap="magma"); axes[1].set_title("RPCA saliency (row mean of S)")
for ax in axes[:2]:
    ax.axis("off")
axes[2].bar(range(3), list(aucs.values()), color=["0.6", "tab:blue", "tab:orange"])
axes[2].set_xticks(range(3)); axes[2].set_xticklabels(list(aucs.keys()), fontsize=7)
axes[2].set_ylim(0.3, 1.0); axes[2].axhline(0.5, color="k", ls=":", lw=1); axes[2].set_ylabel("ROC AUC (supervoxels)")
axes[2].set_title("Which supervoxels contain structure?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With an uneven background, raw brightness is a poor guide (the bright side of the ramp looks
"interesting"). The local-contrast features plus robust PCA separate the common background (low-rank $L$) from the rare,
structured supervoxels (sparse $S$), and the row means of $S$ rank structure-containing supervoxels highly. (With only 15
features, "low rank" is a loose notion here: $L$ keeps several dimensions and $S$ is far from extremely sparse — the
paper's 30 features over 10,000 supervoxels fit the model better.) Ordinary PCA
does **not** do this: its residual mixes noise and signal because a few strong outliers distort the principal directions —
robustness to outliers is the point of *robust* PCA. ⚠️ Honesty note: in this toy, simply averaging the standardized
features already works almost as well as RPCA — most of the gain comes from the local-contrast features. RPCA earns its
keep when the background itself has several structured components.

### 📄 Paper connection — Zhou, Guo, Wang, Zeng, Gao & Xu (2018)

**Paper:** Zhou B, Guo Q, Wang K, Zeng X, Gao X, Xu M. *Feature decomposition based saliency detection in electron
cryo-tomograms.* IEEE BIBM 2018 Workshop on Machine Learning in High Resolution Microscopy, pp. 2467–2473;
[arXiv:1801.10562](https://arxiv.org/abs/1801.10562).

- **Question it answers:** which small fraction of a large tomogram deserves further (expensive) analysis?
- **What the paper actually does:** Gaussian smoothing (σ = 1.8 voxels) → 3D SLIC into n = 10,000 supervoxels (distance
  $D=\sqrt{d_I^2+w(d_s/S)^2m^2}$, $S=\sqrt{N/n}$, $w=0.025$) → 30 features per supervoxel (24 mean 3D-Gabor responses: 4
  orientations about each axis at 2 scales; plus a 6-bin density histogram) → robust PCA $F=L+S$ → saliency = mean of each
  supervoxel's row of $S$ → threshold. ⚠️ It is **not deep learning** (despite appearing among the lab's 2018 papers).
- **What our toy keeps:** smoothing, SLIC supervoxels, per-supervoxel features incl. a 6-bin histogram, principal component
  pursuit, row-mean saliency.
- **What our toy leaves out:** 3D Gabor filters (we use derivative/DoG responses), the paper's scale (10,000 supervoxels),
  real tomograms and human anchor points.
- **Result reported:** the authors report ROC AUCs of about 0.89, 0.86 and 0.82 on three real tomograms (purified yeast 80S
  ribosomes EMPIAR-10045, a rat neuron sub-region, and *Chlamydia trachomatis* type III secretion systems in contact with a
  HeLa cell, EMPIAR-10048), with only 4–11 % of voxels marked salient.
- **Code:** AITom `aitom/segmentation/saliency/feature_decomposition/`.

---
## 6 · Beyond blobs: organelles, and a map of classical pickers

Not everything one wants to find is a compact macromolecule. **Mitochondria** are hundreds of nanometres large, with
internal membranes — DoG at any σ is the wrong tool. Li *et al.* (2019) used a standard object detector from computer
vision instead.

### 📄 Paper connection — Li *et al.* (2019), mitochondria with Faster-RCNN

**Paper:** Li R, Zeng X, Sigmund SE, Lin R, Zhou B, Liu C, Wang K, Jiang R, Freyberg Z, Lv H, Xu M. *Automatic localization
and identification of mitochondria in cellular electron cryo-tomography using faster-RCNN.* BMC Bioinformatics 20(Suppl
3):132 (2019). [doi:10.1186/s12859-019-2650-7](https://doi.org/10.1186/s12859-019-2650-7)

- **Question it answers:** can mitochondria be found and boxed automatically in cryo-ET images of cells?
- **What the paper actually does:** a standard **Faster-RCNN** (ResNet-50 backbone from ImageNet; region-proposal network
  with 9 anchors, NMS at IoU 0.7, 300 proposals; 7 × 7 RoI pooling; log loss + smooth-L1 box loss) on **2D** images; the
  key domain-specific step is preprocessing — **bilateral filtering first, then histogram equalization** (the reverse order
  amplified noise). Output: bounding boxes with class probabilities, not pixel masks. ⚠️ The paper is internally
  inconsistent about whether its 2D inputs are tilt projections or tomogram slices.
- **What our notebook keeps:** only the idea (detection = boxes, a deep detector for large organelles; NMS also appears
  inside Faster-RCNN). Module 17 returns to detection vs segmentation.
- **Result reported:** the authors report AP ≥ 0.95 at IoU ≤ 0.8 with preprocessing (about 0.3 at IoU 0.8 without), and
  per-tomogram F1 of 0.91–1.0 except for one very low-SNR tomogram where detection failed (data: 9 tomograms of rat INS-1E
  cells, 786 images, 402 used for training and 80 for testing).
- **Code:** AITom `aitom/segmentation/detection/organelle/frcnn/` (based on keras-frcnn).

### A map of classical pickers

| Picker | Needs a template? | Needs labels? | Finds | Main failure modes | Module |
|---|---|---|---|---|---|
| Template matching (FLCF × rotations) | **yes** | no | copies of a known shape (+ look-alikes) | template bias, look-alikes, cost ∝ rotations, missing wedge | 10 |
| Difference of Gaussians (Pei 2016) | no (only a size σ) | no | compact blobs of size ≈ √3σ | crowding, membranes, mixed sizes | 10 |
| Saliency + robust PCA (Zhou 2018) | no | no | "unusual" regions | feature design, coarse localization | 10 |
| Template-free detection (Xu 2011) | no | no | recurring patterns | needs many copies | 14 |
| Faster-RCNN (Li 2019) | no | **yes (boxes)** | large organelles | needs annotations, 2D input | 10, 16 |
| PU learning / few-shot / foundation models | no | a few clicks | everything above, learned | domain shift, sparse labels | 23, 24 |

### 📄 Paper connection — book chapter (survey reading)

**Uddin MR, Dip SA, Jha RA, Zeng X, Xu M (2025).** *Feature detection in cryo-electron tomography image analysis.* In
*Cryo-electron Tomography* (Elsevier), pp. 173–215. [doi:10.1016/B978-0-443-18829-9.00008-6](https://doi.org/10.1016/B978-0-443-18829-9.00008-6)
⚠️ We could not access the full text of this chapter while writing; the description below is limited to its title and
indexed keywords. Read the chapter before relying on details. Its indexed keywords (feature detection, image artifacts,
missing data, image noise, subcellular structures and organelles, macromolecular structure, tomographic reconstruction)
match the themes of this module, so it is a natural next reading.

---
## ✅ Checkpoint

1. Why does template matching divide by the *local* standard deviation $\sigma_I(x)$ rather than a global one?
   <details><summary>Answer</summary>So that bright or high-contrast regions (membranes, thick ice, gold beads) cannot get
   high scores just by being bright: the score is a Pearson correlation of the template with the local patch, which only
   rewards matching <i>shape</i>.</details>
2. You run DoG with σ₁ = 9 nm on a tomogram full of RuBisCO (radius ≈ 5–6 nm). What do you expect, and why?
   <details><summary>Answer</summary>Poor recall: DoG is tuned to spheres of radius ≈ √3·σ₁ ≈ 16 nm, so small particles give
   weak responses and neighbouring RuBisCOs merge into one peak. Use σ₁ ≈ 3 nm (or multi-scale DoG).</details>
3. In the threshold rule $T=m+t(M-m)/20$, what happens as $t$ increases?
   <details><summary>Answer</summary>The threshold rises, fewer peaks survive: precision usually goes up and recall goes
   down. $t$ indexes one of 20 equal bins between the smallest (m) and largest (M) DoG peak values.</details>
4. A pick lands 6 nm from the centre of a ribosome whose bounding radius is 15 nm. Is it a true positive under the Pei
   rule? And for a RuBisCO of radius 8 nm?
   <details><summary>Answer</summary>Ribosome: S = 6/15 = 0.4 ≤ 0.5 → yes. RuBisCO: S = 6/8 = 0.75 > 0.5 → no.</details>
5. Your favourite subtomogram matches the ribosome template with score 0.25, higher than any other template. Why is that
   not enough to call it a ribosome, and what would you compute?
   <details><summary>Answer</summary>The best template always wins, even for noise or for a molecule absent from the
   library. Compute a null distribution of scores against many plausible non-ribosome structures (e.g. pseudo-structures,
   Wang et al. 2018) and the Monte Carlo p-value of 0.25 under that null.</details>
6. Why does crowding reduce DoG recall?
   <details><summary>Answer</summary>At the DoG scale, nearby particles blur into each other: their peaks merge or a strong
   neighbour's peak suppresses a weaker one (also through the minimum-distance rule), so some particles never get their own
   peak.</details>

## 🧠 Questions & discussion

1. Template matching with a ribosome template finds ribosomes — and ribosome-like look-alikes. Design an experiment, using
   this notebook's code, that would tell you how often a 26S proteasome is picked as a 20S proteasome.
2. DoG has two knobs (σ₁, t). If you had no ground truth (a real tomogram), how would you choose them?
3. The Wang *et al.* p-value depends on how "plausible" the pseudo-structures are. What goes wrong if they are too simple
   (e.g. spheres)? Too similar to the target?
4. Pickers are often evaluated at the particle level (precision/recall). Why might a biologist care more about *which*
   particles are missed (e.g. those near membranes) than about the overall recall?
5. The missing wedge elongates particles along z. Which of the pickers in this notebook is most affected, and how did we
   already compensate for it in template matching?

## 🚀 Bonus — going further

- **Real data.** Download a tomogram from the [CZ cryoET Data Portal](https://cryoetdataportal.czscience.com/) (e.g. a
  *Mycoplasma pneumoniae* dataset such as EMPIAR-10499), bin it to ~1–2 nm voxels, and run `dog_candidates` with σ₁ ≈
  ribosome radius/√3. Inspect the top 100 picks.
- **AITom.** Run the official DoG tutorial `doc/tutorials/008_particle_picking.py` from
  [AITom](https://github.com/xulabs/aitom) and compare its `peak__partition` + `do_filter` with our `dog_candidates` +
  `remove_close`.
- **Angular accuracy.** Store `arg_rib` (best rotation per voxel) at each true ribosome and compute the angular error to
  the true rotation with the geodesic distance (Module 02). How does it compare with the rotation-set spacing?
- **Better nulls.** Replace `pseudo_structure` with PCA sampling of rotated maps from the zoo (an "image-derived"
  generator closer in spirit to the paper's GAN) and see how the p-values change.
- **Deep dive tutorials:** [`../../LowSNRNoiseProcessing/particle_picking_tutorial.ipynb`](../../LowSNRNoiseProcessing/particle_picking_tutorial.ipynb)
  and [`../../SubtomogramAveraging/`](../../SubtomogramAveraging/).

## 📝 Lab note

```text
Module 11 — date:
Input: in silico cell (4V4R, 3J9I, 6UTJ, 1BXN + vesicle), simulated tomogram (CTF, ±60° wedge, SNR 0.03)
Output: picks + precision/recall/F1; PR curves; p-values for template matches; saliency map
Method: FLCF template matching × rotations + NMS; DoG (σ2 = 1.1 σ1, T = m + t(M − m)/20); pseudo-structure null; RPCA
Key parameter changed & effect:
Failure mode observed (look-alikes? membranes? crowding?):
Paper link: Pei et al. 2016; Wang et al. 2018; Zhou et al. 2018; Li et al. 2019
```

## 📚 References

1. Pei L, Xu M, Frazier Z, Alber F. Simulating cryo electron tomograms of crowded cell cytoplasm for assessment of automated particle picking. *BMC Bioinformatics* 17:405 (2016). doi:10.1186/s12859-016-1283-3
2. Wang KW, Zeng X, Liang X, Huo Z, Xing EP, Xu M. Image-derived generative modeling of pseudo-macromolecular structures – towards the statistical assessment of Electron CryoTomography template matching. BMVC 2018; arXiv:1805.04634.
3. Zhou B, Guo Q, Wang K, Zeng X, Gao X, Xu M. Feature decomposition based saliency detection in electron cryo-tomograms. IEEE BIBM Workshop (MLHRM) 2018, pp. 2467–2473; arXiv:1801.10562.
4. Li R, Zeng X, Sigmund SE, Lin R, Zhou B, Liu C, Wang K, Jiang R, Freyberg Z, Lv H, Xu M. Automatic localization and identification of mitochondria in cellular electron cryo-tomography using faster-RCNN. *BMC Bioinformatics* 20(Suppl 3):132 (2019). doi:10.1186/s12859-019-2650-7
5. Uddin MR, Dip SA, Jha RA, Zeng X, Xu M. Feature detection in cryo-electron tomography image analysis. In *Cryo-electron Tomography*, pp. 173–215, Elsevier (2025). doi:10.1016/B978-0-443-18829-9.00008-6
6. Frangakis AS, Böhm J, Förster F, Nickell S, Nicastro D, Typke D, Hegerl R, Baumeister W. Identification of macromolecular complexes in cryoelectron tomograms of phantom cells. *PNAS* 99:14153–14158 (2002). doi:10.1073/pnas.172520299
7. Roseman AM. Particle finding in electron micrographs using a fast local correlation algorithm. *Ultramicroscopy* 94:225–236 (2003). doi:10.1016/S0304-3991(02)00333-9
8. Beck M, Malmström JA, Lange V, Schmidt A, Deutsch EW, Aebersold R. Visual proteomics of the human pathogen *Leptospira interrogans*. *Nat Methods* 6:817–823 (2009). doi:10.1038/nmeth.1390
9. Candès EJ, Li X, Ma Y, Wright J. Robust principal component analysis? *J ACM* 58:11 (2011). doi:10.1145/1970392.1970395
10. Zeng X, Xu M. AITom: Open-source AI platform for cryo-electron tomography data analysis. arXiv:1911.03044 (2019); <https://github.com/xulabs/aitom>

## ➡️ Next

**[12 · Aligning subtomograms fast: constrained correlation and rotational search](12_fast_subtomogram_alignment.ipynb)** —
now that we have found the particles, we must turn every copy the same way, despite the missing wedge, without trying every
rotation one by one. Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).